<a href="https://colab.research.google.com/github/abhiraj7821/AIML-Projects/blob/main/CIFAR_10_DATASET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets,transforms
from torch.utils.data import DataLoader

class cifar10(nn.Module):
  def __init__(self):
    super(cifar10,self).__init__()
    self.conv1=nn.Conv2d(3,64,kernel_size=5,stride=1,padding=2) #i/p chanel=3 o/p chanel=64 filter size=5 stride=1 padding=2
    #output->32x32x64
    #reLu
    #pool->16x16x64
    self.conv2=nn.Conv2d(64,128,kernel_size=5,stride=1,padding=2) #i/p chanel=64 o/p chanel=128 filter size=5 stride=1 padding=2
    #output->16x16x128
    #relu
    #pool->8x8x128
    self.conv3=nn.Conv2d(128,256,kernel_size=5,stride=1,padding=2) #i/p chanel=128 o/p chanel=256 filter size=5 stride=1 padding=2
    #output->8x8x256
    #relu
    self.conv4=nn.Conv2d(256,256,kernel_size=3,stride=1,padding=1) #i/p chanel=256 o/p chanel=256 filter size=3 stride=1 padding=2
    #output->8x8x256
    #relu
    #pool->4x4x256
    #FLATTEN->4x4x256=4096
    self.fc1=nn.Linear((4*4*256),1024)
    self.fc2=nn.Linear(1024,512)
    self.fc3=nn.Linear(512,10)

  def forward(self,x):
    x=torch.relu(self.conv1(x))
    x=torch.max_pool2d(x,kernel_size=2,stride=2)
    x=torch.relu(self.conv2(x))
    x=torch.max_pool2d(x,kernel_size=2,stride=2)
    x=torch.relu(self.conv3(x))
    x=torch.relu(self.conv4(x))
    x=torch.max_pool2d(x,kernel_size=2,stride=2)
    x=x.view(-1,4*4*256)
    x=torch.relu(self.fc1(x))
    x=torch.relu(self.fc2(x))
    x=torch.softmax(self.fc3(x),dim=1)
    return x


#Data Preparation

transform=transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

train_dataset=datasets.CIFAR10(root='./data',train=True,download=True,transform=transform)
test_dataset=datasets.CIFAR10(root='./data',train=False,download=True,transform=transform)

train_loader=DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=64,shuffle=False)

#model,lossFun,optimizer

model=cifar10()
device=torch.device("cuda" if torch.cuda.is_available() else "CPU")
model=model.to(device)

criterion=nn.CrossEntropyLoss()
optimizer=optim.SGD(model.parameters(),lr=0.01,momentum=0.9)

#training loop

def train(model,device,train_loader,optimizer,criterion,epoch):
  model.train()
  for batch_idx,(data,target) in enumerate(train_loader):
    data,target=data.to(device),target.to(device)
    #forward pss
    output=model(data)
    loss=criterion(output,target)
    #backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if batch_idx%100==0:
      print(f"Train Epoch: {epoch} [{batch_idx*len(data)}/{len(train_loader.dataset)} ({100.*batch_idx/len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}")


#Testing Loop

def test(model,device,test_loader,criterion):
  model.eval()
  test_loss=0
  correct=0
  with torch.no_grad():
    for data,target in test_loader:
      data,target=data.to(device),target.to(device)
      output=model(data)
      #sum up both loss
      test_loss+=criterion(output,target).item()
      #get the index of the max log-probability
      pred=output.argmax(dim=1,keepdim=True)
      correct+=pred.eq(target.view_as(pred)).sum().item()

    test_loss/=len(test_loader.dataset)
    print(f"\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({100.*correct/len(test_loader.dataset):.0f}%)\n")


#Main Training and testing Process
num_epochs=50
for epoch in range(1,num_epochs+1):
  train(model,device,train_loader,optimizer,criterion,epoch)
  test(model,device,test_loader,criterion)

100%|██████████| 170M/170M [00:18<00:00, 9.07MB/s]


Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified
Train Epoch: 1 [0/50000 (0%)]	Loss: 2.302687
Train Epoch: 1 [6400/50000 (13%)]	Loss: 2.302917
Train Epoch: 1 [12800/50000 (26%)]	Loss: 2.302944
Train Epoch: 1 [19200/50000 (38%)]	Loss: 2.302445
Train Epoch: 1 [25600/50000 (51%)]	Loss: 2.302358
Train Epoch: 1 [32000/50000 (64%)]	Loss: 2.302485
Train Epoch: 1 [38400/50000 (77%)]	Loss: 2.301945
Train Epoch: 1 [44800/50000 (90%)]	Loss: 2.302383

Test set: Average loss: 0.0361, Accuracy: 1000/10000 (10%)

Train Epoch: 2 [0/50000 (0%)]	Loss: 2.302143
Train Epoch: 2 [6400/50000 (13%)]	Loss: 2.302640
Train Epoch: 2 [12800/50000 (26%)]	Loss: 2.302033
Train Epoch: 2 [19200/50000 (38%)]	Loss: 2.301068
Train Epoch: 2 [25600/50000 (51%)]	Loss: 2.301138
Train Epoch: 2 [32000/50000 (64%)]	Loss: 2.295972
Train Epoch: 2 [38400/50000 (77%)]	Loss: 2.233374
Train Epoch: 2 [44800/50000 (90%)]	Loss: 2.278170

Test set: Average loss: 0.0350, Accuracy: 2001/10000 (20%)

T